## 1. Crear la sesión de Spark

### ¿Qué haremos?
Crearemos una SparkSession para utilizar Spark desde este notebook.

### ¿Para qué sirve?
La SparkSession es el punto de entrada para trabajar con datos
en Spark. Nos permitirá leer archivos, organizar los datos
en tablas llamadas DataFrames y realizar operaciones sobre ellos.

### Configuración que utilizaremos
- Nombre de la aplicación: s1-verificacion.
- Modo de ejecución: local[*], para utilizar los núcleos
  disponibles dentro del contenedor.
- Puerto interno de Spark UI: 4040, accesible desde Windows
  mediante http://localhost:4042.

### Resultado esperado
Una sesión de Spark activa, lista para leer el CSV indicado
por el docente.

In [4]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("s1-verificacion")
    .master("local[*]")
    .config("spark.ui.port", "4040")
    .getOrCreate()
)

spark

## 2. Cargar un dataset como DataFrame

### ¿Qué haremos?
Leeremos el archivo biblia_ntv_.csv indicado por el docente.

### ¿Qué es un CSV?
Es un archivo que organiza datos en filas y columnas,
separando sus valores mediante un delimitador, como la coma.

### ¿Qué es un DataFrame?
Es una estructura de Spark que organiza los datos en filas
y columnas, similar a una tabla.

### Ubicación del archivo
Dentro del contenedor, el archivo se encuentra en:
/opt/s01-arquitectura/data/biblia_ntv_.csv

### Resultado esperado
Un DataFrame llamado df, con los datos del archivo,
listo para consultar su estructura y mostrar sus filas.

In [5]:
df = spark.read.csv(
    "/opt/s01-arquitectura/data/biblia_ntv_.csv",
    header=True,
    inferSchema=True
)

df

DataFrame[_c0: int, libro: string, capitulo: int, verso: int, texto: string]

## 3. Ver la estructura y las primeras filas

### ¿Qué haremos?
Revisaremos las columnas del DataFrame y mostraremos
las primeras cinco filas del archivo.

### ¿Qué significa revisar la estructura?
Significa identificar el nombre de cada columna y el tipo
de dato que contiene, por ejemplo, números enteros o texto.
Utilizaremos df.printSchema().

### ¿Cómo veremos los registros?
Utilizaremos df.show(5, truncate=False):
- 5 indica que mostraremos las primeras cinco filas.
- truncate=False permite mostrar el contenido completo
  de los textos, sin acortarlos.

### Resultado esperado
La estructura del DataFrame y una tabla con cinco registros.

In [7]:
df.printSchema()

df.show(5, truncate=False)

root
 |-- _c0: integer (nullable = true)
 |-- libro: string (nullable = true)
 |-- capitulo: integer (nullable = true)
 |-- verso: integer (nullable = true)
 |-- texto: string (nullable = true)

+---+-------+--------+-----+-----------------------------------------------------------------------------------------------------------------------------------------------------------------+
|_c0|libro  |capitulo|verso|texto                                                                                                                                                            |
+---+-------+--------+-----+-----------------------------------------------------------------------------------------------------------------------------------------------------------------+
|0  |Génesis|1       |1    |1 El relato de la creación En el principio, Dios creó los cielos y la tierra.                                                                                    |
|1  |Génesis|1       |2    |2 La tierra n

26/11/16 18:58:34 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: , libro, capitulo, verso, texto
 Schema: _c0, libro, capitulo, verso, texto
Expected: _c0 but found: 
CSV file: file:///opt/s01-arquitectura/data/biblia_ntv_.csv


## 4. Transformación simple: seleccionar columnas

### ¿Qué haremos?
Seleccionaremos únicamente las columnas libro, capitulo y verso.

### ¿Para qué sirve?
Permite trabajar con las columnas que necesitamos para un análisis.
En este ejemplo, no mostraremos las columnas _c0 y texto.

### ¿Qué instrucciones utilizaremos?
- select(): selecciona las columnas indicadas.
- show(5): muestra cinco filas del resultado.

Esta operación no modifica el archivo CSV original.

In [8]:
df.select("libro", "capitulo", "verso").show(5)

+-------+--------+-----+
|  libro|capitulo|verso|
+-------+--------+-----+
|Génesis|       1|    1|
|Génesis|       1|    2|
|Génesis|       1|    3|
|Génesis|       1|    4|
|Génesis|       1|    5|
+-------+--------+-----+
only showing top 5 rows


## 3.3 Reconocer el ecosistema de lambda26

### Flujo base del laboratorio

Usuarios → Kafka → Spark Processing → Data Lake / Base de datos
→ Dashboard / Aplicaciones

### ¿Qué representa este flujo?

Describe el recorrido general de los datos:
se generan, se reciben, se procesan, se almacenan
y finalmente se consultan.

### 1. ¿Dónde nacen los eventos?

Los eventos nacen en los programas productores del laboratorio:

- uso-rapido: ec-eventos-py.
- uso-microserv: ec-orden-ms y ec-pago-ms.

Estos programas publican eventos de órdenes y pagos
en los topics orden-eventos y pago-eventos.

### 2. ¿Qué componente ingesta los eventos?

Apache Kafka recibe los eventos enviados por los productores
del laboratorio y los organiza en topics.

Por ejemplo:
- orden-eventos contiene los eventos de órdenes.
- pago-eventos contiene los eventos de pagos.

Los programas consumidores leen esos eventos para utilizarlos.


### 3. ¿Qué componente procesa los datos a escala?

El módulo uso-pyspark utiliza Apache Spark mediante Python
(PySpark) para procesar los datos.

Spark permite trabajar de dos maneras:

- Batch: procesa datos acumulados, por ejemplo, calcular
  el total de ventas del día.
- Streaming: procesa continuamente los eventos que llegan,
  por ejemplo, actualizar el conteo de órdenes.

En esta sesión utilizamos Spark para leer un CSV,
mostrar su estructura y seleccionar columnas.


### 4. ¿Dónde se consumen los resultados?

Los resultados se consultan en dashboards y aplicaciones.

En lambda26:
- Grafana permite visualizar información mediante paneles.
- Prometheus recopila métricas para supervisar el funcionamiento
  del sistema.
- Las bases PostgreSQL de los microservicios, orden_db y pago_db,
  conservan los registros históricos que pueden consultarse.

### Alcance de esta sesión

Hasta este momento ejecutamos únicamente Jupyter y PySpark.
Los demás componentes se reconocen como parte del ecosistema;
todavía no los hemos instalado ni verificado.


## 3.4 Analizar el caso guiado y clasificar batch/streaming

### Caso analizado

En uso-rapido, ec-eventos-py publica y consume eventos
de órdenes mediante Kafka.

En uso-microserv:
- ec-orden-ms publica eventos en orden-eventos.
- ec-pago-ms consume los eventos de órdenes y publica
  eventos en pago-eventos.
- Cada microservicio conserva su histórico en PostgreSQL.

### Tipo de procesamiento seleccionado: ambos

Se propone combinar batch y streaming por estas razones:

1. Streaming: los eventos de órdenes y pagos deben procesarse
   conforme llegan, para dar continuidad al flujo de compra.

2. Batch: los registros históricos permiten realizar análisis
   periódicos, como calcular ventas diarias y comparar periodos.
   Estos análisis se ejecutarán sobre los datos acumulados.

## 3.5 Aplicar la regla de decisión

### Arquitectura seleccionada: Lambda

Según la regla de decisión de la guía, se selecciona Lambda
porque el caso requiere procesamiento histórico y en tiempo real.

La capa batch analizará los registros históricos acumulados.
La capa streaming procesará los nuevos eventos de órdenes y pagos.
La capa de consulta permitirá acceder a los resultados.

## 3.6 Proponer tecnologías y diagrama de flujo

### Tecnologías propuestas

- Kafka: recibe y organiza los eventos de órdenes y pagos.
- Spark Structured Streaming: procesa los eventos conforme llegan.
- Spark Batch: analiza los registros históricos acumulados.
- PostgreSQL / Data Lake: almacena los datos y resultados.
- Grafana: permite consultar resultados mediante paneles.

### Diagrama de flujo propuesto

Productores de órdenes y pagos
              |
              v
            Kafka
              |
       +------+------+
       |             |
       v             v
Spark Streaming   Histórico almacenado
       |             |
       |             v
       |         Spark Batch
       |             |
       +------+------+
              |
              v
   Resultados en PostgreSQL
              |
              v
           Grafana

### Alcance

Este diagrama representa una propuesta de arquitectura Lambda.
En esta sesión solo se han ejecutado Jupyter y PySpark;
los demás componentes todavía no se han implementado.


## 3.7 Ficha de propuesta arquitectónica

| Campo | Propuesta |
|---|---|
| Caso analizado | Sistema de órdenes y pagos de lambda26. Los programas productores generan eventos de órdenes y pagos, y cada microservicio conserva su histórico en PostgreSQL. |
| Tipo de procesamiento | Ambos: streaming para procesar eventos nuevos y batch para elaborar análisis periódicos sobre registros históricos. |
| Arquitectura seleccionada | Lambda, porque se proponen dos caminos de procesamiento separados: uno para eventos nuevos y otro para datos históricos. |
| Diagrama de arquitectura | Productores → Kafka → procesamiento streaming y almacenamiento histórico → procesamiento batch del histórico → resultados consultables → Grafana. El diagrama con las dos ramas se presenta en el apartado 3.6. |
| Tecnologías propuestas | Kafka para ingesta; Spark Structured Streaming y Spark Batch para procesamiento; PostgreSQL / Data Lake para almacenamiento; Grafana para visualización. |
| Supuestos | Se necesitan análisis históricos y atención de eventos nuevos. Los datos incluyen identificadores y fechas que permiten relacionar órdenes con pagos. |
| Riesgos o limitaciones | Pueden llegar eventos duplicados o tardíos. Las capas batch y streaming pueden producir resultados diferentes si aplican reglas distintas. Mantener ambos caminos aumenta la complejidad. |

### Estado de la propuesta

Se verificaron Jupyter y PySpark mediante la lectura de un CSV.
La arquitectura completa es una propuesta y todavía no está
implementada.

## 4. Actividad autónoma: Proyecto Sello

### 4.1 Definición del caso de negocio

**Nombre del proyecto:**
AgroSmart Puno: monitoreo de condiciones de frío y análisis
de riesgo de heladas para cultivos.

**Problema que se propone abordar:**
La información agroclimática puede encontrarse dispersa
y resultar difícil de comparar entre periodos y localidades.
El proyecto busca organizarla para identificar episodios
de frío y apoyar la revisión de condiciones agrícolas.

**Usuarios previstos:**
Productores agrícolas y responsables del seguimiento
agroclimático.

**Dimensiones de análisis:**
- Atmósfera: temperatura del aire y humedad relativa.
- Suelo: temperatura y humedad del suelo, según disponibilidad.

**Fuente de datos propuesta:**
API Open-Meteo. Se documentará la procedencia de cada dato
y se distinguirán los históricos de los pronósticos.

**Alcance inicial:**
Analizar datos correspondientes a Azángaro, Puno.
Si se incorporan otras localidades, sus registros se
identificarán y analizarán por separado.

**Resultados esperados:**
Consultar históricos, identificar condiciones de frío
mediante reglas documentadas y mostrar la antigüedad
de los datos.

**Limitaciones:**
Los datos meteorológicos de modelos no equivalen a mediciones
de sensores instalados en una parcela. Los avisos serán
indicadores para revisión, no confirmaciones de daño al cultivo.
El cultivo y los umbrales deberán definirse antes de evaluar
riesgo específico.

### 4.2 Clasificación del procesamiento

**Tipo seleccionado: ambos, batch y streaming.**

#### Procesamiento batch

Se descargarán datos históricos para analizarlos por lotes.
Esto permitirá calcular temperaturas mínimas diarias,
comparar periodos e identificar episodios de frío.

#### Procesamiento streaming

Un programa consultará periódicamente la API y publicará
los registros nuevos en Kafka. Spark Structured Streaming
los procesará para actualizar indicadores y evaluar reglas
de aviso por condiciones de frío.

#### Justificación

1. El análisis de periodos anteriores requiere procesar
   registros históricos acumulados mediante batch.

2. El seguimiento de nuevos registros requiere un proceso
   continuo que actualice los indicadores conforme
   estén disponibles.

#### Consideración sobre la fuente

Consultar la API con mayor frecuencia no significa que
existan datos nuevos. Se controlarán las fechas de los
registros para evitar publicar duplicados y se distinguirá
la hora del dato de la hora de descarga.

### 4.3 Arquitectura seleccionada

**Arquitectura propuesta: Lambda.**

Se propone una capa batch separada para analizar históricos
agroclimáticos y una capa streaming para procesar registros
nuevos. Ambas ofrecerán resultados consultables para comparar
periodos y monitorear condiciones de frío.

#### Función de las capas

- Batch: calcular resúmenes históricos, como temperaturas
  mínimas diarias y frecuencia de episodios de frío.
- Streaming: actualizar indicadores y evaluar reglas de aviso
  cuando lleguen registros nuevos.
- Serving o consulta: permitir consultar los resultados
  mediante una base de datos y un dashboard.

#### Coherencia entre las capas

Las dos rutas deberán utilizar las mismas unidades,
identificadores de ubicación y reglas de análisis para
evitar resultados contradictorios.

### 4.4 Tecnologías propuestas y diagrama de arquitectura

#### Tecnologías propuestas

| Función | Tecnología | Uso en el proyecto |
|---|---|---|
| Fuente de datos | API Open-Meteo | Obtener datos agroclimáticos por ubicación y fecha. |
| Extracción | Python | Consultar la API y registrar la fecha de descarga. |
| Ingesta de eventos | Kafka | Recibir los registros nuevos publicados por el extractor. |
| Almacenamiento histórico | Archivos JSON y Parquet | Conservar respuestas originales y datos preparados. |
| Procesamiento batch | PySpark | Limpiar históricos y calcular resúmenes por periodo. |
| Procesamiento streaming | Spark Structured Streaming | Procesar registros nuevos y evaluar reglas de aviso. |
| Consulta de resultados | PostgreSQL | Guardar resultados para su consulta. |
| Visualización | Grafana | Mostrar históricos, indicadores y actualidad de los datos. |

#### Diagrama de arquitectura Lambda

                         API Open-Meteo
                               |
                    Extracción con Python
                               |
                +--------------+--------------+
                |                             |
                v                             v
      Históricos en archivos           Registros nuevos
                |                             |
                v                             v
           Spark Batch                      Kafka
                |                             |
                |                             v
                |                  Spark Structured Streaming
                |                             |
                +--------------+--------------+
                               |
                               v
                 Resultados en PostgreSQL
                               |
                               v
                            Grafana

#### Alcance de implementación

En la Unidad 1 se desarrollará la ruta batch conforme
a las guías del docente. La ruta de streaming se incorporará
en la Unidad 2. Las demás herramientas se configurarán
cuando lo requieran las actividades del curso.

### 4.5 Ficha de propuesta arquitectónica

| Campo | Propuesta |
|---|---|
| Caso analizado | AgroSmart Puno: análisis de condiciones agroclimáticas en Azángaro para identificar episodios de frío y apoyar el seguimiento agrícola. Se analizarán variables atmosféricas y del suelo según disponibilidad. |
| Tipo de procesamiento | Ambos: batch para analizar históricos y streaming para procesar registros nuevos obtenidos mediante consultas periódicas a la API. |
| Arquitectura seleccionada | Lambda. Se mantendrán rutas separadas para el análisis histórico y el procesamiento continuo de registros nuevos. |
| Diagrama de arquitectura | Se presenta en el apartado 4.4: extracción desde Open-Meteo, separación en rutas batch y streaming, almacenamiento de resultados y visualización. |
| Tecnologías propuestas | Python, Kafka, PySpark, Spark Structured Streaming, archivos JSON/Parquet, PostgreSQL y Grafana. |
| Supuestos | La API dispone de las variables y periodos necesarios para la ubicación elegida. El equipo puede ejecutar el entorno del laboratorio. Se definirán reglas de análisis y un cultivo antes de evaluar riesgos específicos. |
| Riesgos o limitaciones | Interrupciones o límites de la API; registros duplicados, faltantes o desactualizados; diferencias entre los resultados batch y streaming; recursos limitados del equipo. Los datos de modelos meteorológicos pueden diferir de las condiciones de una parcela. |

### Estado actual

Se verificó el entorno con el CSV indicado por el docente.
La integración con Open-Meteo y las rutas de procesamiento
de AgroSmart Puno todavía no se han implementado.

### 4.6 Error o hallazgo técnico

#### ¿Qué ocurrió?

Al iniciar Spark, apareció el siguiente aviso:

Service 'SparkUI' could not bind on port 4040.
Attempting port 4041.

Spark inició, pero su interfaz de seguimiento utilizó
el puerto interno 4041 porque el 4040 estaba ocupado.

#### ¿Cómo lo identifiqué?

Revisé la salida de la celda que crea la SparkSession.
El aviso indicaba el conflicto y el enlace de Spark UI
mostraba el puerto 4041.

Esto no coincidía con la configuración del laboratorio,
que publica el puerto interno 4040 como localhost:4042.

#### ¿Cómo lo resolví?

Guardé el notebook y reinicié el contenedor desde PowerShell:

docker compose restart

Después actualicé Jupyter y ejecuté la celda de creación
de Spark únicamente en el notebook de trabajo.

#### ¿Cómo comprobé la solución?

La nueva salida mostró Spark UI en el puerto interno 4040,
sin el aviso de puerto ocupado. También pude abrir
http://localhost:4042 desde el navegador.

#### Aprendizaje

Cerrar una pestaña no garantiza que termine su sesión.
Debo controlar las sesiones activas para evitar conflictos
entre notebooks.


### 4.7 Reflexión técnica

Para un sistema IoT que necesite alertas e históricos, propondría Lambda.  
La capa streaming procesaría las lecturas nuevas para detectar condiciones de atención.  
La capa batch analizaría los registros acumulados y permitiría comparar periodos.  
Ambas capas ofrecerían resultados consultables mediante un dashboard.  
Esta separación exige mantener reglas consistentes y aumenta la complejidad.  
Si el histórico pudiera reprocesarse por una sola ruta de streaming, evaluaría Kappa.  
La elección dependería de los requisitos del sistema y de los recursos disponibles.

# Evidencia individual — Sesión 1

- Nombre y apellidos: [completar]
- Equipo: [completar]
- Sesión: S01 — Arquitectura Big Data: Lambda y Kappa,
  batch vs. streaming.
- Rol o aporte realizado: [describir tu aporte real]
- Link del repositorio en GitHub: [completar]